<a href="https://colab.research.google.com/github/GerardLab77/waterbirds-training-dynamics/blob/main/01_baseline_waterbirds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ERM Baseline on WILDS Waterbirds

This notebook trains a simple empirical risk minimisation (ERM) baseline for the WILDS Waterbirds dataset using only images and bird-type labels.

The main prediction target is binary bird type: `0 = landbird`, `1 = waterbird`. Waterbirds also provides metadata describing the background (`land` or `water`) and the bird/background group combination. This metadata is not an input to the model. It is used only after predictions are produced, for subgroup evaluation and CSV export.

## 1. Install Packages and Print Versions

In Google Colab, PyTorch and torchvision are often preinstalled. This cell installs or refreshes the required packages and prints the versions actually imported by the runtime.

In [ ]:
%pip install -q torch torchvision wilds pandas numpy matplotlib tqdm

In [ ]:
import importlib.metadata as importlib_metadata

packages_to_report = ["torch", "torchvision", "wilds", "pandas", "numpy", "matplotlib", "tqdm"]
for package_name in packages_to_report:
    try:
        print(f"{package_name}: {importlib_metadata.version(package_name)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package_name}: NOT INSTALLED")

## 2. Imports, Random Seed, and Device

The seed is fixed for Python, NumPy, and PyTorch. Deterministic settings are enabled where practical, but exact GPU reproducibility can still vary slightly across hardware, CUDA kernels, and package versions.

In [ ]:
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

from wilds import get_dataset
from wilds.common.grouper import CombinatorialGrouper

SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device}")
if device.type == "cuda":
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No CUDA GPU was detected. The notebook will run on CPU and training may be slow. In Colab, choose Runtime > Change runtime type > GPU.")

## 3. Output Folders and Hyperparameters

The pilot run is intentionally short: 5 epochs with a pretrained ResNet-18, AdamW, and standard ImageNet preprocessing.

In [ ]:
OUTPUT_DIRS = {
    "checkpoints": Path("checkpoints"),
    "results": Path("results"),
    "figures": Path("figures"),
}
for output_dir in OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)

HYPERPARAMETERS = {
    "epochs": 5,
    "batch_size": 64,
    "num_workers": 2,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "optimizer": "AdamW",
    "architecture": "torchvision ResNet-18 pretrained on ImageNet",
    "image_size": 224,
}

print(HYPERPARAMETERS)

## 4. Load the Official WILDS Waterbirds Dataset

`get_dataset(dataset="waterbirds", download=True)` asks the WILDS package to download and initialise the Waterbirds dataset. No alternative Waterbirds implementation or manual copy is used here.

`dataset.get_subset(split, transform=...)` returns a `WILDSSubset` containing only the requested official split. For Waterbirds, the standard split names are `train`, `val`, and `test`. The subset keeps the original WILDS indices internally and applies the supplied transform only to the image input.

Each item returned by the dataset or dataloader is a tuple `(x, y, metadata)`:

- `x`: the bird image after the split transform;
- `y`: the bird-type label, where `0 = landbird` and `1 = waterbird`;
- `metadata`: WILDS metadata including background and bird label information used for evaluation only.

In [ ]:
import shutil
import ssl

DATA_ROOT = Path("data")
WATERBIRDS_DIR = DATA_ROOT / "waterbirds_v1.0"

def remove_incomplete_waterbirds_download() -> None:
    if WATERBIRDS_DIR.exists() and not (WATERBIRDS_DIR / "metadata.csv").exists():
        print(f"Removing incomplete Waterbirds download at {WATERBIRDS_DIR}")
        shutil.rmtree(WATERBIRDS_DIR)

def load_waterbirds_with_colab_ssl_fallback():
    remove_incomplete_waterbirds_download()
    try:
        return get_dataset(dataset="waterbirds", root_dir=str(DATA_ROOT), download=True)
    except Exception as first_exc:
        message = str(first_exc)
        if "CERTIFICATE_VERIFY_FAILED" not in message and "metadata.csv" not in message:
            raise
        print("Initial WILDS download failed, likely because the upstream CodaLab SSL certificate is expired.")
        print("Retrying the same official WILDS download with certificate verification disabled for this request.")
        remove_incomplete_waterbirds_download()
        previous_https_context = ssl._create_default_https_context
        ssl._create_default_https_context = ssl._create_unverified_context
        try:
            return get_dataset(dataset="waterbirds", root_dir=str(DATA_ROOT), download=True)
        finally:
            ssl._create_default_https_context = previous_https_context

try:
    dataset = load_waterbirds_with_colab_ssl_fallback()
except Exception as exc:
    raise RuntimeError(
        "Failed to download or load WILDS Waterbirds. The notebook still uses the official WILDS loader, "
        "but the upstream CodaLab-hosted archive may be unavailable from this Colab runtime. "
        "Try Runtime > Restart session and run all, or check https://wilds.stanford.edu/downloads."
    ) from exc

print(f"Dataset name: {dataset.dataset_name}")
print(f"Dataset version: {dataset.version}")
print(f"Split dictionary: {dataset.split_dict}")
print(f"Metadata fields: {dataset.metadata_fields}")
print(f"Number of examples: {len(dataset)}")
print(f"Number of classes: {dataset.n_classes}")

required_splits = {"train", "val", "test"}
missing_splits = required_splits.difference(dataset.split_dict.keys())
assert not missing_splits, f"Missing expected official WILDS splits: {missing_splits}"

required_metadata = {"background", "y"}
missing_metadata = required_metadata.difference(dataset.metadata_fields)
assert not missing_metadata, f"Missing expected Waterbirds metadata fields: {missing_metadata}"

## 5. Transforms, Splits, and DataLoaders

The model uses only transformed image tensors as input. The dataloader also returns labels and metadata, but the training step passes only images to the model and only bird-type labels to cross-entropy loss.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMETERS["image_size"], HYPERPARAMETERS["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMETERS["image_size"], HYPERPARAMETERS["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_data = dataset.get_subset("train", transform=train_transform)
val_data = dataset.get_subset("val", transform=eval_transform)
test_data = dataset.get_subset("test", transform=eval_transform)

train_indices = set(map(int, train_data.indices))
val_indices = set(map(int, val_data.indices))
test_indices = set(map(int, test_data.indices))

assert train_indices.isdisjoint(val_indices), "Train and validation WILDS indices overlap."
assert train_indices.isdisjoint(test_indices), "Train and test WILDS indices overlap."
assert val_indices.isdisjoint(test_indices), "Validation and test WILDS indices overlap."
print("Split index overlap check passed: train, validation, and test indices are disjoint.")
print("Note: prediction CSVs use deterministic split-local sample indices 0..len(split)-1; original WILDS subset indices are also exported for auditing.")

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_data,
    batch_size=HYPERPARAMETERS["batch_size"],
    shuffle=True,
    num_workers=HYPERPARAMETERS["num_workers"],
    pin_memory=(device.type == "cuda"),
    generator=generator,
)
val_loader = DataLoader(
    val_data,
    batch_size=HYPERPARAMETERS["batch_size"],
    shuffle=False,
    num_workers=HYPERPARAMETERS["num_workers"],
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_data,
    batch_size=HYPERPARAMETERS["batch_size"],
    shuffle=False,
    num_workers=HYPERPARAMETERS["num_workers"],
    pin_memory=(device.type == "cuda"),
)

example_images, example_labels, example_metadata = next(iter(train_loader))
assert example_images.ndim == 4 and example_images.shape[1] == 3, "Model input batch should be image tensors only with shape [B, 3, H, W]."
assert example_labels.ndim == 1, "Training labels should be a 1D tensor of bird-type labels."
print(f"Model input batch shape: {tuple(example_images.shape)}")
print(f"Label batch shape: {tuple(example_labels.shape)}")
print(f"Metadata batch shape, kept separate from model inputs: {tuple(example_metadata.shape)}")
print("Leakage check: metadata is returned by the dataloader but is not concatenated to image tensors.")

## 6. Metadata, Group IDs, and Worst-Group Accuracy

Waterbirds metadata includes `background` and `y`. WILDS converts metadata into bird/background group IDs with `CombinatorialGrouper(dataset, ["background", "y"])`. A group therefore corresponds to one of the four combinations: landbird on land, landbird on water, waterbird on land, and waterbird on water.

Worst-group accuracy is calculated by computing accuracy separately within each group and then taking the minimum across groups with at least one example. It is reported alongside overall accuracy because overall accuracy can hide poor performance on rare groups.

In [ ]:
grouper = CombinatorialGrouper(dataset=dataset, groupby_fields=["background", "y"])
metadata_fields = list(dataset.metadata_fields)
background_idx = metadata_fields.index("background")
metadata_y_idx = metadata_fields.index("y")

LABEL_NAMES = {0: "landbird", 1: "waterbird"}
BACKGROUND_NAMES = {0: "land", 1: "water"}

def tensor_to_numpy_1d(value: torch.Tensor) -> np.ndarray:
    return value.detach().cpu().numpy().reshape(-1)

def metadata_to_groups(metadata: torch.Tensor) -> torch.Tensor:
    return grouper.metadata_to_group(metadata.cpu()).long()

def group_name_from_id(group_id: int) -> str:
    return grouper.group_str(int(group_id)).strip()

def compute_group_metrics(preds: torch.Tensor, labels: torch.Tensor, metadata: torch.Tensor) -> Tuple[float, float, Dict[int, Dict[str, float]]]:
    preds = preds.cpu().long()
    labels = labels.cpu().long()
    metadata = metadata.cpu()
    groups = metadata_to_groups(metadata)
    correct = preds.eq(labels)
    overall_acc = correct.float().mean().item()

    per_group: Dict[int, Dict[str, float]] = {}
    for group_id in range(grouper.n_groups):
        mask = groups.eq(group_id)
        count = int(mask.sum().item())
        if count == 0:
            accuracy = float("nan")
        else:
            accuracy = correct[mask].float().mean().item()
        per_group[group_id] = {
            "group_name": group_name_from_id(group_id),
            "count": count,
            "accuracy": accuracy,
        }

    valid_group_accuracies = [row["accuracy"] for row in per_group.values() if row["count"] > 0]
    worst_group_acc = min(valid_group_accuracies) if valid_group_accuracies else float("nan")
    return overall_acc, worst_group_acc, per_group

for split_name, subset in [("train", train_data), ("val", val_data), ("test", test_data)]:
    _, _, per_group = compute_group_metrics(subset.y_array, subset.y_array, subset.metadata_array)
    print(f"{split_name} group sizes:")
    for group_id, row in per_group.items():
        print(f"  group {group_id}: {row['group_name']}, n={row['count']}")

## 7. Model, Training, Evaluation, and Export Helpers

The classifier is an ImageNet-pretrained ResNet-18 with the final fully connected layer replaced for two bird classes. Training uses cross-entropy loss on bird-type labels only.

In [ ]:
def build_model() -> nn.Module:
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
    except AttributeError:
        model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

model = build_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMETERS["learning_rate"],
    weight_decay=HYPERPARAMETERS["weight_decay"],
)

print(model.__class__.__name__)
print(f"Final classification layer: {model.fc}")
print("Training loss check: CrossEntropyLoss(logits, bird_type_labels) uses labels only, not metadata.")

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.Module, device: torch.device) -> float:
    model.train()
    running_loss = 0.0
    total_examples = 0

    for images, labels, metadata in tqdm(loader, desc="Training", leave=False):
        assert images.ndim == 4 and images.shape[1] == 3, "Only image tensors should be passed to the model."
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_examples += batch_size

    return running_loss / total_examples

@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, torch.Tensor]:
    model.eval()
    all_probs: List[torch.Tensor] = []
    all_preds: List[torch.Tensor] = []
    all_labels: List[torch.Tensor] = []
    all_metadata: List[torch.Tensor] = []

    for images, labels, metadata in tqdm(loader, desc="Evaluating", leave=False):
        assert images.ndim == 4 and images.shape[1] == 3, "Only image tensors should be passed to the model."
        images = images.to(device, non_blocking=True)
        labels = labels.long()

        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu()
        preds = probs.argmax(dim=1)

        all_probs.append(probs)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
        all_metadata.append(metadata.cpu())

    outputs = {
        "probs": torch.cat(all_probs),
        "preds": torch.cat(all_preds),
        "labels": torch.cat(all_labels),
        "metadata": torch.cat(all_metadata),
    }
    assert len(outputs["preds"]) == len(outputs["labels"]) == len(outputs["metadata"]), "Prediction, label, and metadata counts must match."
    print("Leakage check: validation/test metadata was collected only after image-only model predictions were produced.")
    return outputs

def evaluate_model(model: nn.Module, loader: DataLoader, split_name: str, device: torch.device) -> Dict[str, object]:
    outputs = collect_predictions(model, loader, device)
    overall_acc, worst_group_acc, per_group = compute_group_metrics(outputs["preds"], outputs["labels"], outputs["metadata"])
    print(f"{split_name} overall accuracy: {overall_acc:.4f}")
    print(f"{split_name} worst-group accuracy: {worst_group_acc:.4f}")
    return {
        "overall_accuracy": overall_acc,
        "worst_group_accuracy": worst_group_acc,
        "per_group": per_group,
        "outputs": outputs,
    }

def make_prediction_dataframe(outputs: Dict[str, torch.Tensor], subset) -> pd.DataFrame:
    probs = outputs["probs"].cpu()
    preds = outputs["preds"].cpu().long()
    labels = outputs["labels"].cpu().long()
    metadata = outputs["metadata"].cpu().long()
    groups = metadata_to_groups(metadata)

    assert len(preds) == len(subset), "The number of exported predictions must match the split size."
    assert len(preds) == len(subset.indices), "The number of exported predictions must match the available WILDS subset indices."

    df = pd.DataFrame({
        "split_local_sample_index": np.arange(len(preds), dtype=int),
        "wilds_original_index": np.asarray(subset.indices, dtype=int),
        "true_bird_label": tensor_to_numpy_1d(labels).astype(int),
        "true_bird_label_name": [LABEL_NAMES[int(value)] for value in labels],
        "predicted_bird_label": tensor_to_numpy_1d(preds).astype(int),
        "predicted_bird_label_name": [LABEL_NAMES[int(value)] for value in preds],
        "prob_landbird": tensor_to_numpy_1d(probs[:, 0]),
        "prob_waterbird": tensor_to_numpy_1d(probs[:, 1]),
        "predicted_confidence": tensor_to_numpy_1d(probs.max(dim=1).values),
        "correct": tensor_to_numpy_1d(preds.eq(labels)).astype(int),
        "group_id": tensor_to_numpy_1d(groups).astype(int),
        "group_name": [group_name_from_id(int(value)) for value in groups],
        "background_label": tensor_to_numpy_1d(metadata[:, background_idx]).astype(int),
        "background_name": [BACKGROUND_NAMES[int(value)] for value in metadata[:, background_idx]],
        "metadata_y": tensor_to_numpy_1d(metadata[:, metadata_y_idx]).astype(int),
    })
    return df

def per_group_dataframe(split_name: str, per_group: Dict[int, Dict[str, float]]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "split": split_name,
            "group_id": group_id,
            "group_name": row["group_name"],
            "group_size": row["count"],
            "accuracy": row["accuracy"],
        }
        for group_id, row in per_group.items()
    ])

## 8. Train the ERM Baseline

At the end of every epoch, the notebook reports mean training loss, validation overall accuracy, and validation worst-group accuracy.

In [ ]:
metrics_rows: List[Dict[str, float]] = []

for epoch in range(1, HYPERPARAMETERS["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate_model(model, val_loader, "validation", device)

    row = {
        "epoch": epoch,
        "training_loss": train_loss,
        "validation_overall_accuracy": val_metrics["overall_accuracy"],
        "validation_worst_group_accuracy": val_metrics["worst_group_accuracy"],
    }
    metrics_rows.append(row)
    print(
        f"Epoch {epoch:02d}/{HYPERPARAMETERS['epochs']} | "
        f"train loss={train_loss:.4f} | "
        f"val acc={val_metrics['overall_accuracy']:.4f} | "
        f"val worst-group acc={val_metrics['worst_group_accuracy']:.4f}"
    )

training_metrics_df = pd.DataFrame(metrics_rows)
training_metrics_df.to_csv("results/training_metrics.csv", index=False)
print("Saved results/training_metrics.csv")

## 9. Final Validation and Test Evaluation

After training, predictions are collected for validation and test. Metadata is then used to calculate per-group and worst-group accuracy and to export CSV files.

In [ ]:
final_val_metrics = evaluate_model(model, val_loader, "final validation", device)
final_test_metrics = evaluate_model(model, test_loader, "final test", device)

print("\nFinal validation metrics")
print(f"Overall accuracy: {final_val_metrics['overall_accuracy']:.4f}")
print(f"Worst-group accuracy: {final_val_metrics['worst_group_accuracy']:.4f}")

print("\nFinal test metrics")
print(f"Overall accuracy: {final_test_metrics['overall_accuracy']:.4f}")
print(f"Worst-group accuracy: {final_test_metrics['worst_group_accuracy']:.4f}")

val_group_df = per_group_dataframe("val", final_val_metrics["per_group"])
test_group_df = per_group_dataframe("test", final_test_metrics["per_group"])
all_group_df = pd.concat([val_group_df, test_group_df], ignore_index=True)

print("\nValidation per-group accuracy and group sizes")
display(val_group_df)
print("\nTest per-group accuracy and group sizes")
display(test_group_df)

val_predictions_df = make_prediction_dataframe(final_val_metrics["outputs"], val_data)
test_predictions_df = make_prediction_dataframe(final_test_metrics["outputs"], test_data)

assert len(val_predictions_df) == len(val_data), "Validation export row count does not match validation split size."
assert len(test_predictions_df) == len(test_data), "Test export row count does not match test split size."
print("Export count check passed: validation and test prediction CSV row counts match their split sizes.")

val_predictions_df.to_csv("results/val_predictions.csv", index=False)
test_predictions_df.to_csv("results/test_predictions.csv", index=False)
print("Saved results/val_predictions.csv")
print("Saved results/test_predictions.csv")

## 10. Checkpoint and Training Curves

The checkpoint stores the model state, optimizer state, final epoch, hyperparameters, random seed, and package versions. The figure shows training loss and validation accuracy across epochs.

In [ ]:
package_versions = {}
for package_name in packages_to_report:
    try:
        package_versions[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package_name] = "not installed"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": HYPERPARAMETERS["epochs"],
    "hyperparameters": HYPERPARAMETERS,
    "random_seed": SEED,
    "package_versions": package_versions,
    "label_names": LABEL_NAMES,
    "metadata_fields": metadata_fields,
}
torch.save(checkpoint, "checkpoints/resnet18_waterbirds_erm_5epochs.pt")
print("Saved checkpoints/resnet18_waterbirds_erm_5epochs.pt")

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(training_metrics_df["epoch"], training_metrics_df["training_loss"], marker="o", color="tab:blue", label="Training loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_xticks(training_metrics_df["epoch"])

ax2 = ax1.twinx()
ax2.plot(training_metrics_df["epoch"], training_metrics_df["validation_overall_accuracy"], marker="s", color="tab:green", label="Validation accuracy")
ax2.plot(training_metrics_df["epoch"], training_metrics_df["validation_worst_group_accuracy"], marker="^", color="tab:red", label="Validation worst-group accuracy")
ax2.set_ylabel("Accuracy", color="tab:green")
ax2.tick_params(axis="y", labelcolor="tab:green")
ax2.set_ylim(0, 1)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
plt.title("Waterbirds ERM Baseline Training Curves")
plt.tight_layout()
plt.savefig("figures/baseline_training_curves.png", dpi=200)
plt.show()
print("Saved figures/baseline_training_curves.png")

## 11. WILDS API Assumptions to Manually Verify

Before treating the numbers as final, manually verify these assumptions in the runtime output:

- `get_dataset(dataset="waterbirds", download=True)` successfully loads the WILDS Waterbirds dataset from the WILDS package.
- `dataset.split_dict` contains the official `train`, `val`, and `test` splits.
- `dataset.metadata_fields` contains `background` and `y`; recent WILDS versions may also append `from_source_domain`.
- Waterbirds labels follow the WILDS convention used in the dataset source: `0 = landbird`, `1 = waterbird`.
- Background metadata follows the WILDS convention used in the dataset source: `0 = land`, `1 = water`.
- `CombinatorialGrouper(dataset, ["background", "y"])` produces the four intended bird/background groups.

The notebook exports split-local sample indices and also exports WILDS subset indices. If a future WILDS release exposes separate globally unique original sample IDs, those could be added to the CSVs.

## Interpretation

A model can achieve high overall accuracy while performing badly on rare shortcut-conflicting groups because overall accuracy is dominated by common groups. In Waterbirds, a classifier can learn patterns that work well for the majority bird/background combinations while failing on examples where the bird type conflicts with the background cue.

Worst-group accuracy is needed alongside overall accuracy because it asks how well the model performs on the hardest bird/background group, rather than averaging that group away.

This notebook is only the ERM baseline for a later study of per-example training dynamics. It does not implement forgetting events, confidence variability, subgroup discovery, ranking, reweighting, GroupDRO, JTT, or any mitigation method.

Strong or weak baseline performance alone does not prove that the model is causally relying on background shortcuts. It shows an outcome pattern that motivates deeper analysis, not a causal mechanism by itself.

## Run Summary

Files created or saved by this notebook:

- `checkpoints/resnet18_waterbirds_erm_5epochs.pt`
- `results/val_predictions.csv`
- `results/test_predictions.csv`
- `results/training_metrics.csv`
- `figures/baseline_training_curves.png`

Main hyperparameters:

- Model: ImageNet-pretrained torchvision ResNet-18
- Classes: 2 bird-type classes, landbird and waterbird
- Epochs: 5
- Batch size: 64
- Optimizer: AdamW
- Learning rate: 1e-4
- Weight decay: 1e-4
- Image size: 224 x 224
- Seed: 42

Steps to rerun in Google Colab:

1. Open this notebook in Colab.
2. Choose `Runtime > Change runtime type > GPU`.
3. Run all cells from top to bottom.
4. Confirm package versions, selected device, split sizes, and metadata fields in the printed output.
5. Inspect the final validation/test metrics, exported CSVs, checkpoint, and training curve figure.

Assumptions and limitations to verify before trusting the results:

- The WILDS Waterbirds download URL is available during the run.
- The installed WILDS version still uses `background` and `y` as Waterbirds metadata fields.
- The label and background integer conventions match the printed metadata mapping.
- The short 5-epoch run is a pilot baseline, not a tuned final model.
- GPU operations may vary slightly even with deterministic settings enabled.